# Lithium Supply Chain — Network Model (all 6 stages)

**Mine -> Processing -> Cathode -> Cell -> Pack -> Recycling**, regions
**US / CHN / AUS**, years from `index.csv` (2026-2036).

Extends the validated 2-stage (`mine`, `proc`) model from
`Edited_RealData_networkModel_v3.ipynb` to all six stages, using the exact
same node/arc encoding, balance/conversion/capacity pattern -- nothing
about how nodes or arcs are defined has changed, it's just applied five
more times. Two structural consequences of extending the chain (not
arbitrary changes):

- `proc`, `cath`, `cell` no longer carry their own `demand` term -- they're
  intermediate stages now that the chain continues past them, so (like
  `mine` already does) they feed the next stage's `consume` instead. Only
  `pack` (real end-consumer demand) and `rec` (a recycling target) keep a
  `demand` term.
- `RATIO` becomes stage-scoped (`RATIO[stage][tech]`) instead of one flat
  dict, because `rec` produces the same tech names (`loh`/`lce`) `proc`
  does, at a different (and currently placeholder) recovery rate -- a flat
  `RATIO` would silently let one collide with the other.

The capacity/investment block (production bounded by capacity, investment
lag, active-unit dynamics, initial conditions, max new units per year) is
already written generically over `N` and `STAGES` in the 2-stage version --
it needed zero code changes here, only bigger `N`/`STAGES`/`max_cap`/
`init_capacity`.

## Imports

In [ ]:
import gurobipy as gp
from gurobipy import GRB, quicksum
import pandas as pd

## License

Wired in before the model is defined, as requested.

In [ ]:
import os

# Gurobi licence. WLS credentials are read from the environment if they are
# set; otherwise Gurobi's own licence file (gurobi.lic) is used, which is what
# an ordinary academic named-user licence installs.
#
# The fallback matters: explicit WLS params passed to gp.Env(params=...)
# OVERRIDE the licence file entirely. Demanding the env vars unconditionally
# made these notebooks unrunnable on a machine that had a perfectly good
# licence -- they failed with KeyError before reaching the model.
_wls = {
    "WLSACCESSID": os.environ.get("GUROBI_WLSACCESSID"),
    "WLSSECRET": os.environ.get("GUROBI_WLSSECRET"),
    "LICENSEID": os.environ.get("GUROBI_LICENSEID"),
}
if all(_wls.values()):
    _wls["LICENSEID"] = int(_wls["LICENSEID"])
    env = gp.Env(params=_wls)
else:
    env = gp.Env()


## Sets

In [ ]:
from pathlib import Path
import os

# Repo-relative data location (was a Google Drive path). Override with the
# MINERALNET_DATA env var if the instance tables live elsewhere. DATA_DIR keeps
# a trailing separator so `DATA_DIR + 'file.csv'` still works throughout.
_data_base = Path(os.environ.get(
    'MINERALNET_DATA',
    Path.cwd().parent / 'data' / 'raw' / 'input_data_3region',
))
DATA_DIR = os.path.join(str(_data_base), '')

Index = pd.read_csv(DATA_DIR + 'index.csv')


In [ ]:
MINE_T = ['spod', 'brine', 'clay']
PROC_T = ['loh', 'lce']
CATH_T = ['nmc', 'lfp']
CELL_T = ['GWh_nmc', 'GWh_lfp']
PACK_T = ['bev_nmc', 'bev_lfp', 'phev_nmc', 'phev_lfp']
REC_T  = ['lce', 'loh']   # NEW: recycling recovers the same materials proc does

STAGES = ['mine', 'proc', 'cath', 'cell', 'pack', 'rec']

# RATIO is stage-scoped: rec's output techs (lce/loh) share names with
# proc's, at a different rate, so a flat dict would collide between them.
# proc/cath/cell/pack values are the same physical constants test8.2 used
# (lce_li, loh_li, GWh_nmc_kt_li, GWh_lfp_kt_li, GWh_bev, GWh_phev), just
# re-expressed as "output per unit input consumed" to match this model's
# `prod == RATIO[stage][tech] * consume` convention. rec's yield is new --
# no equivalent existed in test8.2 -- and is PLACEHOLDER pending a real
# recycling-yield study.
RATIO = {
    'proc': {'loh': 3.45101, 'lce': 5.324},
    'cath': {'nmc': 1 / 3.45101, 'lfp': 1 / 5.324},
    'cell': {'GWh_nmc': 8.11, 'GWh_lfp': 10.58},
    'pack': {'bev_nmc': 1 / 62, 'phev_nmc': 1 / 15, 'bev_lfp': 1 / 62, 'phev_lfp': 1 / 15},
    'rec':  {'loh': 30, 'lce': 35},
}

In [ ]:
R = Index['Region'].dropna().tolist()
Y = Index['Year'].dropna().tolist()

# Y_start must cover a full 5 years from Y's own start, or active_dynamics /
# capacity_accum reach for invest[n, y-5] / new_capacity[n, y-5] one year
# before Y begins. Computed from min(Y) instead of hardcoded so this stays
# correct if Y's range ever changes.
Y_start = [y for y in Y if y < min(Y) + 5]      # first 5 years (seeded)
Y_warm  = [y for y in Y if y >= min(Y) + 5]     # years where lag terms y-5,y-1 valid

In [ ]:
Y, R

## NODES & ARCS

In [ ]:
N_mine = [('mine', t, r) for t in MINE_T for r in R]
N_proc = [('proc', tm, tp, r) for tm in MINE_T for tp in PROC_T for r in R]
N_cath = [('cath', tp, tc, r) for tp in PROC_T for tc in CATH_T for r in R]
N_cell = [('cell', tc, tl, r) for tc in CATH_T for tl in CELL_T for r in R]
N_pack = [('pack', tl, tk, r) for tl in CELL_T for tk in PACK_T for r in R]
N_rec  = [('rec', tk, tg, r) for tk in PACK_T for tg in REC_T for r in R]
N = N_mine + N_proc + N_cath + N_cell + N_pack + N_rec

mine_trade = [('mine', t, ri, rj) for t in MINE_T for ri in R for rj in R if ri != rj]
proc_trade = [('proc', tm, tp, ri, rj) for tm in MINE_T for tp in PROC_T for ri in R for rj in R if ri != rj]
cath_trade = [('cath', tp, tc, ri, rj) for tp in PROC_T for tc in CATH_T for ri in R for rj in R if ri != rj]
cell_trade = [('cell', tc, tl, ri, rj) for tc in CATH_T for tl in CELL_T for ri in R for rj in R if ri != rj]
pack_trade = [('pack', tl, tk, ri, rj) for tl in CELL_T for tk in PACK_T for ri in R for rj in R if ri != rj]
rec_trade  = [('rec', tk, tg, ri, rj) for tk in PACK_T for tg in REC_T for ri in R for rj in R if ri != rj]

MINE_ARCS = list(range(len(mine_trade)))
PROC_ARCS = list(range(len(proc_trade)))
CATH_ARCS = list(range(len(cath_trade)))
CELL_ARCS = list(range(len(cell_trade)))
PACK_ARCS = list(range(len(pack_trade)))
REC_ARCS  = list(range(len(rec_trade)))

In [ ]:
mine_in = {n: [] for n in N_mine}; mine_out = {n: [] for n in N_mine}
for i, a in enumerate(mine_trade):
    mine_out[('mine', a[1], a[2])].append(i)
    mine_in[('mine', a[1], a[3])].append(i)

proc_in = {n: [] for n in N_proc}; proc_out = {n: [] for n in N_proc}
for i, a in enumerate(proc_trade):
    proc_out[('proc', a[1], a[2], a[3])].append(i)
    proc_in[('proc', a[1], a[2], a[4])].append(i)

cath_in = {n: [] for n in N_cath}; cath_out = {n: [] for n in N_cath}
for i, a in enumerate(cath_trade):
    cath_out[('cath', a[1], a[2], a[3])].append(i)
    cath_in[('cath', a[1], a[2], a[4])].append(i)

cell_in = {n: [] for n in N_cell}; cell_out = {n: [] for n in N_cell}
for i, a in enumerate(cell_trade):
    cell_out[('cell', a[1], a[2], a[3])].append(i)
    cell_in[('cell', a[1], a[2], a[4])].append(i)

pack_in = {n: [] for n in N_pack}; pack_out = {n: [] for n in N_pack}
for i, a in enumerate(pack_trade):
    pack_out[('pack', a[1], a[2], a[3])].append(i)
    pack_in[('pack', a[1], a[2], a[4])].append(i)

rec_in = {n: [] for n in N_rec}; rec_out = {n: [] for n in N_rec}
for i, a in enumerate(rec_trade):
    rec_out[('rec', a[1], a[2], a[3])].append(i)
    rec_in[('rec', a[1], a[2], a[4])].append(i)

## Read Files

In [ ]:
# DATA_DIR is set in the data-import cell above (repo-relative Path).


In [ ]:
# Demand
mine_demand = pd.read_csv(DATA_DIR + 'mine_demand.csv')
proc_demand = pd.read_csv(DATA_DIR + 'proc_demand.csv')
cath_demand = pd.read_csv(DATA_DIR + 'cath_demand.csv')
cell_demand = pd.read_csv(DATA_DIR + 'cell_demand.csv')
pack_demand = pd.read_csv(DATA_DIR + 'pack_demand.csv')
rec_demand  = pd.read_csv(DATA_DIR + 'rec_demand.csv')

In [ ]:
# Mine Cost
mine_prod_cost  = pd.read_csv(DATA_DIR + 'mine_prod_cost.csv')
mine_var_cost   = pd.read_csv(DATA_DIR + 'mine_var_cost.csv')
mine_fixed_cost = pd.read_csv(DATA_DIR + 'mine_fixed_cost.csv')

mine_prod_CO2  = pd.read_csv(DATA_DIR + 'mine_prod_CO2.csv')
mine_var_CO2   = pd.read_csv(DATA_DIR + 'mine_var_CO2.csv')
mine_fixed_CO2 = pd.read_csv(DATA_DIR + 'mine_fixed_CO2.csv')

In [ ]:
# Processing Cost
proc_prod_cost  = pd.read_csv(DATA_DIR + 'proc_prod_cost.csv')
proc_var_cost   = pd.read_csv(DATA_DIR + 'proc_var_cost.csv')
proc_fixed_cost = pd.read_csv(DATA_DIR + 'proc_fixed_cost.csv')

proc_prod_CO2  = pd.read_csv(DATA_DIR + 'proc_prod_CO2.csv')
proc_var_CO2   = pd.read_csv(DATA_DIR + 'proc_var_CO2.csv')
proc_fixed_CO2 = pd.read_csv(DATA_DIR + 'proc_fixed_CO2.csv')

In [ ]:
# Cathode Cost
cath_prod_cost  = pd.read_csv(DATA_DIR + 'cath_prod_cost.csv')
cath_var_cost   = pd.read_csv(DATA_DIR + 'cath_var_cost.csv')
cath_fixed_cost = pd.read_csv(DATA_DIR + 'cath_fixed_cost.csv')

cath_prod_CO2  = pd.read_csv(DATA_DIR + 'cath_prod_CO2.csv')
cath_var_CO2   = pd.read_csv(DATA_DIR + 'cath_var_CO2.csv')
cath_fixed_CO2 = pd.read_csv(DATA_DIR + 'cath_fixed_CO2.csv')

In [ ]:
# Cell Cost
cell_prod_cost  = pd.read_csv(DATA_DIR + 'cell_prod_cost.csv')
cell_var_cost   = pd.read_csv(DATA_DIR + 'cell_var_cost.csv')
cell_fixed_cost = pd.read_csv(DATA_DIR + 'cell_fixed_cost.csv')

cell_prod_CO2  = pd.read_csv(DATA_DIR + 'cell_prod_CO2.csv')
cell_var_CO2   = pd.read_csv(DATA_DIR + 'cell_var_CO2.csv')
cell_fixed_CO2 = pd.read_csv(DATA_DIR + 'cell_fixed_CO2.csv')

In [ ]:
# Pack Cost
pack_prod_cost  = pd.read_csv(DATA_DIR + 'pack_prod_cost.csv')
pack_var_cost   = pd.read_csv(DATA_DIR + 'pack_var_cost.csv')
pack_fixed_cost = pd.read_csv(DATA_DIR + 'pack_fixed_cost.csv')

pack_prod_CO2  = pd.read_csv(DATA_DIR + 'pack_prod_CO2.csv')
pack_var_CO2   = pd.read_csv(DATA_DIR + 'pack_var_CO2.csv')
pack_fixed_CO2 = pd.read_csv(DATA_DIR + 'pack_fixed_CO2.csv')

In [ ]:
# Recycling Cost
rec_prod_cost  = pd.read_csv(DATA_DIR + 'rec_prod_cost.csv')
rec_var_cost   = pd.read_csv(DATA_DIR + 'rec_var_cost.csv')
rec_fixed_cost = pd.read_csv(DATA_DIR + 'rec_fixed_cost.csv')

rec_prod_CO2  = pd.read_csv(DATA_DIR + 'rec_prod_CO2.csv')
rec_var_CO2   = pd.read_csv(DATA_DIR + 'rec_var_CO2.csv')
rec_fixed_CO2 = pd.read_csv(DATA_DIR + 'rec_fixed_CO2.csv')

## PARAMETERS

Same no-averaging approach validated for mine/proc: every cost is keyed on
the CSV's real dimensions -- `(tech, region, year)` for mine, `(input,
output, region, year)` for everything else. No import/export/shipping CSV
exists in the data, so `alpha`/`import_coef`/`ship_cost` stay as modeling
assumptions, same as before.

In [ ]:
alpha       = {'US': 1.3, 'CHN': 1.0, 'AUS': 1.1}   # regional cost multiplier
import_coef = {'US': 0.5, 'CHN': 2.0, 'AUS': 0.3}   # import cost coefficient
ship_cost   = {'US': 80,  'CHN': 60,  'AUS': 50}     # shipping $/kt by importer

In [ ]:
# --- Mine production / variable / fixed cost -----------------------------
mine_prod_window = mine_prod_cost[mine_prod_cost.Year.isin(Y) & mine_prod_cost.Technology.isin(MINE_T)]
prod_cost_mine = {(row.Technology, row.Region, row.Year): row.Cost for row in mine_prod_window.itertuples()}

mine_var_window = mine_var_cost[mine_var_cost.Year.isin(Y) & mine_var_cost.Technology.isin(MINE_T)]
var_cost_mine = {(row.Technology, row.Region, row.Year): row.Cost for row in mine_var_window.itertuples()}

mine_fixed_window = mine_fixed_cost[mine_fixed_cost.Year.isin(Y) & mine_fixed_cost.Technology.isin(MINE_T)]
fixed_cost_mine = {(row.Technology, row.Region, row.Year): row.Cost for row in mine_fixed_window.itertuples()}

In [ ]:
# --- Generic helper for the (Input, Output, Region, Year)-keyed stages ---
def io_cost_dict(df, input_set, output_set):
    window = df[df.Year.isin(Y) & df.Input.isin(input_set) & df.Output.isin(output_set)]
    return {(row.Input, row.Output, row.Region, row.Year): row.Cost for row in window.itertuples()}

In [ ]:
# --- Processing production / variable / fixed cost ------------------------
prod_cost_proc  = io_cost_dict(proc_prod_cost,  MINE_T, PROC_T)
var_cost_proc   = io_cost_dict(proc_var_cost,   MINE_T, PROC_T)
fixed_cost_proc = io_cost_dict(proc_fixed_cost, MINE_T, PROC_T)

# --- Cathode production / variable / fixed cost ----------------------------
prod_cost_cath  = io_cost_dict(cath_prod_cost,  PROC_T, CATH_T)
var_cost_cath   = io_cost_dict(cath_var_cost,   PROC_T, CATH_T)
fixed_cost_cath = io_cost_dict(cath_fixed_cost, PROC_T, CATH_T)

# --- Cell production / variable / fixed cost --------------------------------
prod_cost_cell  = io_cost_dict(cell_prod_cost,  CATH_T, CELL_T)
var_cost_cell   = io_cost_dict(cell_var_cost,   CATH_T, CELL_T)
fixed_cost_cell = io_cost_dict(cell_fixed_cost, CATH_T, CELL_T)

# --- Pack production / variable / fixed cost --------------------------------
prod_cost_pack  = io_cost_dict(pack_prod_cost,  CELL_T, PACK_T)
var_cost_pack   = io_cost_dict(pack_var_cost,   CELL_T, PACK_T)
fixed_cost_pack = io_cost_dict(pack_fixed_cost, CELL_T, PACK_T)

# --- Recycling production / variable / fixed cost ---------------------------
# rec_prod_cost.csv carries an extra 'Variable' label column the others
# don't; it doesn't affect the (Input,Output,Region,Year) key, so it's
# simply ignored (not dropped -- itertuples never reads it).
prod_cost_rec  = io_cost_dict(rec_prod_cost,  PACK_T, REC_T)
var_cost_rec   = io_cost_dict(rec_var_cost,   PACK_T, REC_T)
fixed_cost_rec = io_cost_dict(rec_fixed_cost, PACK_T, REC_T)

### Demand

Only `pack` (real end-consumer demand) and `rec` (a recycling target) get
a `demand` term now that the chain continues past `proc`/`cath`/`cell` --
see the note at the top of this notebook. Same adjustable `region_share`
pattern as before: real CSV values, scaled so you can rebalance demand
across regions without touching the CSV (all three regions currently carry
identical demand in the data, so `region_share` should sum to 1 to keep
each year's CSV total unchanged).

In [ ]:
region_share = {'US': 1 / 3, 'CHN': 1 / 3, 'AUS': 1 / 3}   # adjust freely; must sum to 1


def region_share_demand(df, tech_set):
    window = df[df.Year.isin(Y) & df.Technology.isin(tech_set) & (df.Region == 'US')]
    return {
        (row.Technology, r, row.Year): region_share[r] * row.Demand
        for row in window.itertuples()
        for r in R
    }


demand_pack = region_share_demand(pack_demand, PACK_T)
demand_rec  = region_share_demand(rec_demand, REC_T)

### Capacity & investment

`max_cap`/`max_new_units` match test8.2's own constants (`max_cap_mine`,
`max_cap_proc`, `max_cap_cath`, `max_cap_cell`, `max_cap_pack`,
`max_cap_rec`, `max_new_units`) exactly. No capacity CSV exists in the
data, so `init_active`/`init_capacity` stay as assumptions. `init_active`
is a single scalar shared by every stage (same structure as the validated
2-stage model), so it has to satisfy the most demanding stage's
`capacity / max_cap` ratio -- here that's `pack` (`max_cap['pack'] == 1`),
sized to clear real 2036 pack demand (~25 bev_nmc + ~9 phev_nmc + ~11
bev_lfp + ~4 phev_lfp per region) with margin.

In [ ]:
max_cap = {
    'mine': 25, 'proc': 100, 'cath': 12, 'cell': 100, 'pack': 1, 'rec': 15,
}   # test8.2's max_cap_mine / max_cap_proc / max_cap_cath / max_cap_cell / max_cap_pack / max_cap_rec
max_new_units = 25   # test8.2's max_new_units

init_active = 80   # operating units at start (shared by every stage)
init_capacity = {
    'mine': 1000, 'proc': 2500, 'cath': 500, 'cell': 2500, 'pack': 80, 'rec': 250,
}   # kt (or MM units, for pack) initial capacity per stage

## MODEL

In [ ]:
m = gp.Model('lithium_network', env=env)
m.Params.OutputFlag = 0

### Variables

In [ ]:
flow_mine = {(i, y): m.addVar(lb=0, name=f'flow_mine_{i}_{y}') for i in MINE_ARCS for y in Y}
flow_proc = {(i, y): m.addVar(lb=0, name=f'flow_proc_{i}_{y}') for i in PROC_ARCS for y in Y}
flow_cath = {(i, y): m.addVar(lb=0, name=f'flow_cath_{i}_{y}') for i in CATH_ARCS for y in Y}
flow_cell = {(i, y): m.addVar(lb=0, name=f'flow_cell_{i}_{y}') for i in CELL_ARCS for y in Y}
flow_pack = {(i, y): m.addVar(lb=0, name=f'flow_pack_{i}_{y}') for i in PACK_ARCS for y in Y}
flow_rec  = {(i, y): m.addVar(lb=0, name=f'flow_rec_{i}_{y}')  for i in REC_ARCS  for y in Y}

prod         = {(n, y): m.addVar(lb=0, name=f'prod_{n}_{y}')                      for n in N for y in Y}
capacity     = {(n, y): m.addVar(lb=0, name=f'capacity_{n}_{y}')                  for n in N for y in Y}
new_capacity = {(n, y): m.addVar(lb=0, name=f'new_capacity_{n}_{y}')              for n in N for y in Y}
invest       = {(n, y): m.addVar(lb=0, vtype=GRB.INTEGER, name=f'invest_{n}_{y}') for n in N for y in Y}
active       = {(n, y): m.addVar(lb=0, vtype=GRB.INTEGER, name=f'active_{n}_{y}') for n in N for y in Y}

# one consume dict per stage transition (mine feeds proc, proc feeds cath, ...)
consume_proc = {(tm, tp, r, y): m.addVar(lb=0, name=f'consume_proc_{tm}_{tp}_{r}_{y}')
                for tm in MINE_T for tp in PROC_T for r in R for y in Y}
consume_cath = {(tp, tc, r, y): m.addVar(lb=0, name=f'consume_cath_{tp}_{tc}_{r}_{y}')
                for tp in PROC_T for tc in CATH_T for r in R for y in Y}
consume_cell = {(tc, tl, r, y): m.addVar(lb=0, name=f'consume_cell_{tc}_{tl}_{r}_{y}')
                for tc in CATH_T for tl in CELL_T for r in R for y in Y}
consume_pack = {(tl, tk, r, y): m.addVar(lb=0, name=f'consume_pack_{tl}_{tk}_{r}_{y}')
                for tl in CELL_T for tk in PACK_T for r in R for y in Y}
consume_rec  = {(tk, tg, r, y): m.addVar(lb=0, name=f'consume_rec_{tk}_{tg}_{r}_{y}')
                for tk in PACK_T for tg in REC_T for r in R for y in Y}
m.update()

## CONSTRAINTS

### Balance Constraint

In [ ]:
## Mine: production + imports == exports + consumed by conversion into proc
mine_balance = m.addConstrs(
    (prod[('mine', tm, r), y] + quicksum(flow_mine[a, y] for a in mine_in[('mine', tm, r)])
     == quicksum(flow_mine[a, y] for a in mine_out[('mine', tm, r)])
        + quicksum(consume_proc[tm, tp, r, y] for tp in PROC_T)
     for tm in MINE_T for r in R for y in Y), name='mine_balance')

## Proc: production + imports == exports + consumed by conversion into cath
## (no longer a terminal demand stage now that cath/cell/pack/rec continue the chain)
proc_balance = m.addConstrs(
    (quicksum(prod[('proc', tm, tp, r), y] for tm in MINE_T)
     + quicksum(flow_proc[a, y] for tm in MINE_T for a in proc_in[('proc', tm, tp, r)])
     == quicksum(flow_proc[a, y] for tm in MINE_T for a in proc_out[('proc', tm, tp, r)])
        + quicksum(consume_cath[tp, tc, r, y] for tc in CATH_T)
     for tp in PROC_T for r in R for y in Y), name='proc_balance')

## Cath: production + imports == exports + consumed by conversion into cell
cath_balance = m.addConstrs(
    (quicksum(prod[('cath', tp, tc, r), y] for tp in PROC_T)
     + quicksum(flow_cath[a, y] for tp in PROC_T for a in cath_in[('cath', tp, tc, r)])
     == quicksum(flow_cath[a, y] for tp in PROC_T for a in cath_out[('cath', tp, tc, r)])
        + quicksum(consume_cell[tc, tl, r, y] for tl in CELL_T)
     for tc in CATH_T for r in R for y in Y), name='cath_balance')

## Cell: production + imports == exports + consumed by conversion into pack
cell_balance = m.addConstrs(
    (quicksum(prod[('cell', tc, tl, r), y] for tc in CATH_T)
     + quicksum(flow_cell[a, y] for tc in CATH_T for a in cell_in[('cell', tc, tl, r)])
     == quicksum(flow_cell[a, y] for tc in CATH_T for a in cell_out[('cell', tc, tl, r)])
        + quicksum(consume_pack[tl, tk, r, y] for tk in PACK_T)
     for tl in CELL_T for r in R for y in Y), name='cell_balance')

## Pack: production + imports == exports + demand + consumed by recycling
## (pack is the real end-consumer stage AND rec's upstream feed, same as networkModel_v3)
pack_balance = m.addConstrs(
    (quicksum(prod[('pack', tl, tk, r), y] for tl in CELL_T)
     + quicksum(flow_pack[a, y] for tl in CELL_T for a in pack_in[('pack', tl, tk, r)])
     == quicksum(flow_pack[a, y] for tl in CELL_T for a in pack_out[('pack', tl, tk, r)])
        + demand_pack[(tk, r, y)]
        + quicksum(consume_rec[tk, tg, r, y] for tg in REC_T)
     for tk in PACK_T for r in R for y in Y), name='pack_balance')

## Rec: production + imports == exports + demand (a recycling target, terminal)
rec_balance = m.addConstrs(
    (quicksum(prod[('rec', tk, tg, r), y] for tk in PACK_T)
     + quicksum(flow_rec[a, y] for tk in PACK_T for a in rec_in[('rec', tk, tg, r)])
     == quicksum(flow_rec[a, y] for tk in PACK_T for a in rec_out[('rec', tk, tg, r)])
        + demand_rec[(tg, r, y)]
     for tg in REC_T for r in R for y in Y), name='rec_balance')

### Conversion Constraint

In [ ]:
conversion_proc = m.addConstrs(
    (prod[('proc', tm, tp, r), y] == RATIO['proc'][tp] * consume_proc[tm, tp, r, y]
     for tm in MINE_T for tp in PROC_T for r in R for y in Y), name='conversion_proc')

conversion_cath = m.addConstrs(
    (prod[('cath', tp, tc, r), y] == RATIO['cath'][tc] * consume_cath[tp, tc, r, y]
     for tp in PROC_T for tc in CATH_T for r in R for y in Y), name='conversion_cath')

conversion_cell = m.addConstrs(
    (prod[('cell', tc, tl, r), y] == RATIO['cell'][tl] * consume_cell[tc, tl, r, y]
     for tc in CATH_T for tl in CELL_T for r in R for y in Y), name='conversion_cell')

conversion_pack = m.addConstrs(
    (prod[('pack', tl, tk, r), y] == RATIO['pack'][tk] * consume_pack[tl, tk, r, y]
     for tl in CELL_T for tk in PACK_T for r in R for y in Y), name='conversion_pack')

conversion_rec = m.addConstrs(
    (prod[('rec', tk, tg, r), y] == RATIO['rec'][tg] * consume_rec[tk, tg, r, y]
     for tk in PACK_T for tg in REC_T for r in R for y in Y), name='conversion_rec')

### Demand Satisfaction Constraint

In [ ]:
# Global demand: total production == total demand, per tech per year --
# redundant with rec_balance once trade is closed (imports of one region are
# exports of another), same as the validated 2-stage model's redundant
# check for proc. NOTE: there's no equivalent for pack -- pack also feeds
# consume_rec (recycling), so "total pack production == total pack demand"
# would directly contradict pack_balance (which requires production to
# cover demand *plus* consume_rec) the moment consume_rec is nonzero. An
# earlier draft of this cell had that constraint and it made the model
# infeasible for exactly this reason.
global_demand_rec = m.addConstrs(
    (quicksum(prod[('rec', tk, tg, r), y] for tk in PACK_T for r in R)
     == quicksum(demand_rec[(tg, r, y)] for r in R)
     for tg in REC_T for y in Y), name='global_demand_rec')

### Capacity and Investment Constraint

Unchanged from the validated 2-stage model -- already generic over `N`/
`STAGES`, so extending to 6 stages needed no edits here at all.

In [ ]:
# 5. Production bounded by capacity
prod_capacity = m.addConstrs(
    (prod[n, y] <= capacity[n, y] for n in N for y in Y), name='prod_capacity')

# 6. New capacity and investment
invest_capacity = m.addConstrs(
    (new_capacity[n, y] <= max_cap[n[0]] * invest[n, y] for n in N for y in Y), name='invest_capacity')

# 7. Active unit count with 5-year build lag
active_dynamics = m.addConstrs(
    (active[n, y] == active[n, y - 1] + invest[n, y - 5] for n in N for y in Y_warm), name='active_dynamics')

# 8. Capacity bounded by active units
capacity_logical = m.addConstrs(
    (capacity[n, y] <= max_cap[n[0]] * active[n, y] for n in N for y in Y), name='capacity_logical')

# 9. Capacity accumulation with 5-year lag
capacity_accum = m.addConstrs(
    (capacity[n, y] <= new_capacity[n, y - 5] + capacity[n, y - 1] for n in N for y in Y_warm), name='capacity_accum')

# 10. Initial capacity and investment (first 5 years)
# == not <= : Y_start is a known starting condition (test8.2's own initial
# conditions are exact equalities too), not just a cap.
init_active_constr = m.addConstrs(
    (active[n, y] == init_active for n in N for y in Y_start), name='init_active')
init_capacity_constr = m.addConstrs(
    (capacity[n, y] == init_capacity[n[0]] for n in N for y in Y_start), name='init_capacity')

# 11. Max new units per stage per year
max_units = m.addConstrs(
    (quicksum(invest[n, y] for n in N if n[0] == s) <= max_new_units for s in STAGES for y in Y), name='max_units')

## OBJECTIVE

In [ ]:
def stage_cost_constr(stage_name, nodes, prod_cost, var_cost, fixed_cost, is_mine=False):
    stage_cost = m.addVars(R, lb=0, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name=f'{stage_name}_cost')
    if is_mine:
        cost_expr = {
            r: alpha[r] * (
                quicksum(prod[('mine', t, r), y] * prod_cost[(t, r, y)] for y in Y for t in MINE_T) +
                quicksum(new_capacity[('mine', t, r), y] * var_cost[(t, r, y)] for y in Y for t in MINE_T) +
                quicksum(invest[('mine', t, r), y] * fixed_cost[(t, r, y)] for y in Y for t in MINE_T)
            )
            for r in R
        }
    else:
        input_t, output_t = nodes
        cost_expr = {
            r: alpha[r] * (
                quicksum(prod[(stage_name, ti, to, r), y] * prod_cost[(ti, to, r, y)]
                         for y in Y for ti in input_t for to in output_t) +
                quicksum(new_capacity[(stage_name, ti, to, r), y] * var_cost[(ti, to, r, y)]
                         for y in Y for ti in input_t for to in output_t) +
                quicksum(invest[(stage_name, ti, to, r), y] * fixed_cost[(ti, to, r, y)]
                         for y in Y for ti in input_t for to in output_t)
            )
            for r in R
        }
    stage_cost_constr_ = m.addConstrs((stage_cost[r] == cost_expr[r] for r in R), name=f'{stage_name}_cost_constr')
    return stage_cost, stage_cost_constr_


mine_cost, mine_cost_constr = stage_cost_constr('mine', None, prod_cost_mine, var_cost_mine, fixed_cost_mine, is_mine=True)
proc_cost, proc_cost_constr = stage_cost_constr('proc', (MINE_T, PROC_T), prod_cost_proc, var_cost_proc, fixed_cost_proc)
cath_cost, cath_cost_constr = stage_cost_constr('cath', (PROC_T, CATH_T), prod_cost_cath, var_cost_cath, fixed_cost_cath)
cell_cost, cell_cost_constr = stage_cost_constr('cell', (CATH_T, CELL_T), prod_cost_cell, var_cost_cell, fixed_cost_cell)
pack_cost, pack_cost_constr = stage_cost_constr('pack', (CELL_T, PACK_T), prod_cost_pack, var_cost_pack, fixed_cost_pack)
rec_cost, rec_cost_constr   = stage_cost_constr('rec', (PACK_T, REC_T), prod_cost_rec, var_cost_rec, fixed_cost_rec)

In [ ]:
import_cost = m.addVars(R, lb=0, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name='import_cost')
import_cost_constr = m.addConstrs((
    import_cost[r] == import_coef[r] * (
        quicksum(ship_cost[r] * flow_mine[i, y] for i in MINE_ARCS if mine_trade[i][3] == r for y in Y) +
        quicksum(ship_cost[r] * flow_proc[i, y] for i in PROC_ARCS if proc_trade[i][4] == r for y in Y) +
        quicksum(ship_cost[r] * flow_cath[i, y] for i in CATH_ARCS if cath_trade[i][4] == r for y in Y) +
        quicksum(ship_cost[r] * flow_cell[i, y] for i in CELL_ARCS if cell_trade[i][4] == r for y in Y) +
        quicksum(ship_cost[r] * flow_pack[i, y] for i in PACK_ARCS if pack_trade[i][4] == r for y in Y) +
        quicksum(ship_cost[r] * flow_rec[i, y]  for i in REC_ARCS  if rec_trade[i][4] == r  for y in Y)
    )
    for r in R), name='import_cost_constr')

total_cost = m.addVars(R, lb=0, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name='total_cost')
total_cost_constr = m.addConstrs((
    total_cost[r] == mine_cost[r] + proc_cost[r] + cath_cost[r] + cell_cost[r] + pack_cost[r] + rec_cost[r] + import_cost[r]
    for r in R), name='total_cost_constr')

m.setObjective(total_cost.sum(), GRB.MINIMIZE)

## Solution

In [ ]:
m.Params.TimeLimit = 120
m.optimize()
print("STATUS:", m.Status, "(2=OPTIMAL)  Vars:", m.NumVars, " Constrs:", m.NumConstrs)
if m.Status in (GRB.OPTIMAL, GRB.TIME_LIMIT):
    print(f"Objective: ${m.ObjVal:,.0f}\n")
    for r in R:
        print(f"  {r}: mine={mine_cost[r].X:,.0f} proc={proc_cost[r].X:,.0f} cath={cath_cost[r].X:,.0f} "
              f"cell={cell_cost[r].X:,.0f} pack={pack_cost[r].X:,.0f} rec={rec_cost[r].X:,.0f} "
              f"import={import_cost[r].X:,.0f} total={total_cost[r].X:,.0f}")

    sample_year = Y[0]
    print(f"\nMine balance {sample_year}:")
    for r in R:
        for tm in MINE_T:
            p = prod[('mine', tm, r), sample_year].X
            imp = sum(flow_mine[a, sample_year].X for a in mine_in[('mine', tm, r)])
            exp = sum(flow_mine[a, sample_year].X for a in mine_out[('mine', tm, r)])
            cons = sum(consume_proc[tm, tp, r, sample_year].X for tp in PROC_T)
            print(f"  {tm} {r}: bal={p + imp - exp - cons:.4f}")

    print(f"\nPack balance {sample_year}:")
    for r in R:
        for tk in PACK_T:
            p = sum(prod[('pack', tl, tk, r), sample_year].X for tl in CELL_T)
            imp = sum(flow_pack[a, sample_year].X for tl in CELL_T for a in pack_in[('pack', tl, tk, r)])
            exp = sum(flow_pack[a, sample_year].X for tl in CELL_T for a in pack_out[('pack', tl, tk, r)])
            cons = sum(consume_rec[tk, tg, r, sample_year].X for tg in REC_T)
            bal = p + imp - exp - demand_pack[(tk, r, sample_year)] - cons
            print(f"  {tk} {r}: bal={bal:.4f}")

    print(f"\nRec balance {sample_year}:")
    for r in R:
        for tg in REC_T:
            p = sum(prod[('rec', tk, tg, r), sample_year].X for tk in PACK_T)
            imp = sum(flow_rec[a, sample_year].X for tk in PACK_T for a in rec_in[('rec', tk, tg, r)])
            exp = sum(flow_rec[a, sample_year].X for tk in PACK_T for a in rec_out[('rec', tk, tg, r)])
            print(f"  {tg} {r}: bal={p + imp - exp - demand_rec[(tg, r, sample_year)]:.4f}")
else:
    print("Model did not reach an optimal or time-limited solution.")

## Results

Same layout as the 2-stage model's Results section, extended to all six
stages: one node-variable table per stage, one consume table per stage
transition (5 of them now instead of 1), trade flows combined across all
six stages' arcs, and cost by region across all six stage-cost columns.

In [ ]:
assert m.Status in (GRB.OPTIMAL, GRB.TIME_LIMIT)

STAGE_ORDER = ['mine', 'proc', 'cath', 'cell', 'pack', 'rec']
STAGE_INPUT_T = {'proc': MINE_T, 'cath': PROC_T, 'cell': CATH_T, 'pack': CELL_T, 'rec': PACK_T}
STAGE_OUTPUT_T = {'mine': MINE_T, 'proc': PROC_T, 'cath': CATH_T, 'cell': CELL_T, 'pack': PACK_T, 'rec': REC_T}
STAGE_INPUT_COL = {'proc': 'Mine_Tech', 'cath': 'Proc_Tech', 'cell': 'Cath_Tech', 'pack': 'Cell_Tech', 'rec': 'Pack_Tech'}
STAGE_OUTPUT_COL = {'mine': 'Mine_Tech', 'proc': 'Proc_Tech', 'cath': 'Cath_Tech', 'cell': 'Cell_Tech', 'pack': 'Pack_Tech', 'rec': 'Rec_Tech'}
STAGE_NODES = {'mine': N_mine, 'proc': N_proc, 'cath': N_cath, 'cell': N_cell, 'pack': N_pack, 'rec': N_rec}
STAGE_TRADE = {
    'mine': (mine_trade, flow_mine), 'proc': (proc_trade, flow_proc), 'cath': (cath_trade, flow_cath),
    'cell': (cell_trade, flow_cell), 'pack': (pack_trade, flow_pack), 'rec': (rec_trade, flow_rec),
}
STAGE_CONSUME = {
    'proc': (consume_proc, MINE_T, PROC_T), 'cath': (consume_cath, PROC_T, CATH_T),
    'cell': (consume_cell, CATH_T, CELL_T), 'pack': (consume_pack, CELL_T, PACK_T),
    'rec': (consume_rec, PACK_T, REC_T),
}

# --- Node variables: prod, capacity, new_capacity, invest, active, one table per stage ---
stage_var_tables = {}
for s in STAGE_ORDER:
    rows = []
    for n in STAGE_NODES[s]:
        for y in Y:
            row = {'Year': y}
            if s == 'mine':
                row[STAGE_OUTPUT_COL[s]] = n[1]
                row['Region'] = n[2]
            else:
                row[STAGE_INPUT_COL[s]] = n[1]
                row[STAGE_OUTPUT_COL[s]] = n[2]
                row['Region'] = n[3]
            row.update({
                'Production': round(prod[n, y].X, 2),
                'Capacity': round(capacity[n, y].X, 2),
                'New_Capacity': round(new_capacity[n, y].X, 2),
                'Invest_units': int(round(invest[n, y].X)),
                'Active_units': int(round(active[n, y].X)),
            })
            rows.append(row)
    stage_var_tables[s] = pd.DataFrame(rows)

df_mine = stage_var_tables['mine']
df_proc = stage_var_tables['proc']
df_cath = stage_var_tables['cath']
df_cell = stage_var_tables['cell']
df_pack = stage_var_tables['pack']
df_rec = stage_var_tables['rec']

In [ ]:
# --- Consume variables: one table per stage transition ---
stage_consume_tables = {}
for s, (consume_dict, in_t, out_t) in STAGE_CONSUME.items():
    rows = []
    for ti in in_t:
        for to in out_t:
            for r in R:
                for y in Y:
                    v = consume_dict[ti, to, r, y].X
                    if v > 0.01:
                        rows.append({
                            'Year': y, STAGE_INPUT_COL[s]: ti, STAGE_OUTPUT_COL[s]: to,
                            'Region': r, 'Consumed': round(v, 2),
                        })
    stage_consume_tables[s] = pd.DataFrame(rows)

df_consume_proc = stage_consume_tables['proc']
df_consume_cath = stage_consume_tables['cath']
df_consume_cell = stage_consume_tables['cell']
df_consume_pack = stage_consume_tables['pack']
df_consume_rec = stage_consume_tables['rec']

In [ ]:
# --- Trade flows, combined across all six stages ---
trade_rows = []
for s in STAGE_ORDER:
    trade, flow = STAGE_TRADE[s]
    for i, a in enumerate(trade):
        for y in Y:
            v = flow[i, y].X
            if v > 0.01:
                if s == 'mine':
                    tech_label, exporter, importer = a[1], a[2], a[3]
                else:
                    tech_label, exporter, importer = f'{a[1]}->{a[2]}', a[3], a[4]
                trade_rows.append({'Year': y, 'Stage': s, 'Tech': tech_label,
                                    'Exporter': exporter, 'Importer': importer, 'Flow': round(v, 2)})
df_trade = pd.DataFrame(trade_rows)

In [ ]:
# --- Cost by region ---
cost_rows = [{
    'Region': r,
    'Mine_$': round(mine_cost[r].X), 'Proc_$': round(proc_cost[r].X), 'Cath_$': round(cath_cost[r].X),
    'Cell_$': round(cell_cost[r].X), 'Pack_$': round(pack_cost[r].X), 'Rec_$': round(rec_cost[r].X),
    'Import_$': round(import_cost[r].X), 'Total_$': round(total_cost[r].X),
} for r in R]
df_cost = pd.DataFrame(cost_rows)

print('df_mine, df_proc, df_cath, df_cell, df_pack, df_rec, df_consume_*, df_trade, df_cost ready')
df_cost

In [ ]:
with pd.ExcelWriter('lithium_results.xlsx', engine='openpyxl') as w:
    df_mine.to_excel(w, sheet_name='Mine_Variables', index=False)
    df_proc.to_excel(w, sheet_name='Proc_Variables', index=False)
    df_cath.to_excel(w, sheet_name='Cath_Variables', index=False)
    df_cell.to_excel(w, sheet_name='Cell_Variables', index=False)
    df_pack.to_excel(w, sheet_name='Pack_Variables', index=False)
    df_rec.to_excel(w, sheet_name='Rec_Variables', index=False)
    df_consume_proc.to_excel(w, sheet_name='Consume_Proc', index=False)
    df_consume_cath.to_excel(w, sheet_name='Consume_Cath', index=False)
    df_consume_cell.to_excel(w, sheet_name='Consume_Cell', index=False)
    df_consume_pack.to_excel(w, sheet_name='Consume_Pack', index=False)
    df_consume_rec.to_excel(w, sheet_name='Consume_Rec', index=False)
    df_trade.to_excel(w, sheet_name='Trade_Flows', index=False)
    df_cost.to_excel(w, sheet_name='Cost_by_Region', index=False)
print('Saved lithium_results.xlsx')
# from google.colab import files
# files.download('lithium_results.xlsx')

## Model Validation

One balance sheet per stage. Each intermediate stage (`proc`/`cath`/`cell`)
checks production against what the *next* stage actually consumed
(`consume_*`, the real decision variable, not a reconstruction via
`prod/RATIO`) -- aggregated over the input-tech dimension per output tech,
matching exactly how `proc_balance`/`cath_balance`/`cell_balance` are
themselves written (they collapse the input-tech sum inside the
constraint). `pack` checks against both real consumer demand *and* what
`rec` consumed from it -- the one stage with two simultaneous downstream
claims. `mine` and `rec` check against `consume_proc` and `demand_rec`
respectively, same idea as the validated 2-stage model's `Mine_Balance`/
`Proc_Balance`.

In [ ]:
def balance_table(stage, output_t, input_t, flow, arcs_in, arcs_out, target_fn, target_label):
    rows = []
    for to in output_t:
        for r in R:
            for y in Y:
                production = sum(prod[(stage, ti, to, r), y].X for ti in input_t)
                imp = sum(flow[a, y].X for ti in input_t for a in arcs_in[(stage, ti, to, r)])
                exp = sum(flow[a, y].X for ti in input_t for a in arcs_out[(stage, ti, to, r)])
                target = target_fn(to, r, y)
                net = production + imp - exp
                rows.append({
                    'Year': y, STAGE_OUTPUT_COL[stage]: to, 'Region': r,
                    'Production': round(production, 3), 'Import': round(imp, 3), 'Export': round(exp, 3),
                    target_label: round(target, 3), 'Net(P+I-E)': round(net, 3),
                    'Balance(Net-Target)': round(net - target, 3),
                })
    return pd.DataFrame(rows)


# --- Mine balance sheet (mine's node shape differs: (mine,tech,region), no input tech) ---
mine_val = []
for tm in MINE_T:
    for r in R:
        for y in Y:
            production = prod[('mine', tm, r), y].X
            imp = sum(flow_mine[a, y].X for a in mine_in[('mine', tm, r)])
            exp = sum(flow_mine[a, y].X for a in mine_out[('mine', tm, r)])
            downstream = sum(consume_proc[tm, tp, r, y].X for tp in PROC_T)
            bal = production + imp - exp - downstream
            mine_val.append({'Year': y, 'Mine_Tech': tm, 'Region': r,
                              'Production': round(production, 3), 'Import': round(imp, 3), 'Export': round(exp, 3),
                              'Downstream_Consumed': round(downstream, 3), 'Balance(P+I-E-D)': round(bal, 3)})
df_mine_val = pd.DataFrame(mine_val)

# --- Proc / Cath / Cell balance sheets: production vs. what the next stage consumed ---
df_proc_val = balance_table('proc', PROC_T, MINE_T, flow_proc, proc_in, proc_out,
                             lambda to, r, y: sum(consume_cath[to, tc, r, y].X for tc in CATH_T),
                             'Downstream_Consumed')
df_cath_val = balance_table('cath', CATH_T, PROC_T, flow_cath, cath_in, cath_out,
                             lambda to, r, y: sum(consume_cell[to, tl, r, y].X for tl in CELL_T),
                             'Downstream_Consumed')
df_cell_val = balance_table('cell', CELL_T, CATH_T, flow_cell, cell_in, cell_out,
                             lambda to, r, y: sum(consume_pack[to, tk, r, y].X for tk in PACK_T),
                             'Downstream_Consumed')

# --- Pack balance sheet: production vs. real demand PLUS what rec consumed ---
df_pack_val = balance_table('pack', PACK_T, CELL_T, flow_pack, pack_in, pack_out,
                             lambda to, r, y: demand_pack[(to, r, y)] + sum(consume_rec[to, tg, r, y].X for tg in REC_T),
                             'Demand_Plus_Downstream')

# --- Rec balance sheet: production vs. its own demand target (terminal, no further downstream) ---
df_rec_val = balance_table('rec', REC_T, PACK_T, flow_rec, rec_in, rec_out,
                            lambda to, r, y: demand_rec[(to, r, y)],
                            'Demand')

In [ ]:
with pd.ExcelWriter('lithium_validation.xlsx', engine='openpyxl') as w:
    df_mine_val.to_excel(w, sheet_name='Mine_Balance', index=False)
    df_proc_val.to_excel(w, sheet_name='Proc_Balance', index=False)
    df_cath_val.to_excel(w, sheet_name='Cath_Balance', index=False)
    df_cell_val.to_excel(w, sheet_name='Cell_Balance', index=False)
    df_pack_val.to_excel(w, sheet_name='Pack_Balance', index=False)
    df_rec_val.to_excel(w, sheet_name='Rec_Balance', index=False)
print('Saved lithium_validation.xlsx')

for name, df in [('Mine', df_mine_val), ('Proc', df_proc_val), ('Cath', df_cath_val),
                  ('Cell', df_cell_val), ('Pack', df_pack_val), ('Rec', df_rec_val)]:
    col = 'Balance(P+I-E-D)' if name == 'Mine' else 'Balance(Net-Target)'
    print(f'Max |{name} imbalance| (should be ~0):', df[col].abs().max())

# from google.colab import files
# files.download('lithium_validation.xlsx')
df_pack_val[df_pack_val.Year == Y[0]]

## Network Diagram

Same drawing approach as the 2-stage model's `draw_network` (region-colored
circles, conversion arrows between adjacent stages, dashed trade arrows
within a stage), generalized from 2 columns to all 6. Per-arc value labels
are dropped here (a 6-stage diagram has far more arcs than the 2-stage
one, and labeling all of them would just be clutter) -- node value labels
and colors carry the same information the 2-stage version's labels did.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

REGION_COLOR = {'US': '#3B6FB0', 'CHN': '#C0392B', 'AUS': '#27AE60'}
STAGE_LABEL = {'mine': 'MINE', 'proc': 'PROCESSING', 'cath': 'CATHODE',
                'cell': 'CELL', 'pack': 'PACK', 'rec': 'RECYCLING'}


def stage_output_value(stage, tech, r, year):
    """total production of `tech` at `stage`, summed over whichever input feeds it."""
    if stage == 'mine':
        return prod[('mine', tech, r), year].X
    return sum(prod[(stage, ti, tech, r), year].X for ti in STAGE_INPUT_T[stage])


def draw_network(year, min_flow=0.1, save=None):
    fig, ax = plt.subplots(figsize=(26, 9))
    ax.set_aspect('equal')

    col_gap = 6.0
    stage_x = {s: i * col_gap for i, s in enumerate(STAGE_ORDER)}
    row_gap = 1.0
    stage_nodes = {s: [(s, t, r) for r in R for t in STAGE_OUTPUT_T[s]] for s in STAGE_ORDER}
    n_rows = max(len(v) for v in stage_nodes.values())
    RAD = 0.16

    pos = {}
    for s in STAGE_ORDER:
        nodes = stage_nodes[s]
        offset = (n_rows - len(nodes)) / 2.0
        for idx, n in enumerate(nodes):
            pos[n] = (stage_x[s], (len(nodes) - 1 - idx) * row_gap + offset * row_gap)

    # conversion arcs: stage i's output feeds stage i+1 as input, same region
    for i in range(len(STAGE_ORDER) - 1):
        s_from, s_to = STAGE_ORDER[i], STAGE_ORDER[i + 1]
        for t_from in STAGE_OUTPUT_T[s_from]:
            for t_to in STAGE_OUTPUT_T[s_to]:
                for r in R:
                    v = prod[(s_to, t_from, t_to, r), year].X
                    if v > min_flow:
                        x1, y1 = pos[(s_from, t_from, r)]
                        x2, y2 = pos[(s_to, t_to, r)]
                        ax.annotate('', xy=(x2 - RAD, y2), xytext=(x1 + RAD, y1),
                                    arrowprops=dict(arrowstyle='-|>', color='#555', lw=1.0, alpha=0.5))

    # trade arcs (curved, within a stage, region to region)
    for s in STAGE_ORDER:
        trade, flow = STAGE_TRADE[s]
        for i, a in enumerate(trade):
            v = flow[i, year].X
            if v > min_flow:
                if s == 'mine':
                    tech, ri, rj = a[1], a[2], a[3]
                else:
                    tech, ri, rj = a[2], a[3], a[4]   # output tech identifies the traded commodity
                x1, y1 = pos[(s, tech, ri)]
                x2, y2 = pos[(s, tech, rj)]
                rad = 0.3 if y2 > y1 else -0.3
                ax.annotate('', xy=(x2 + RAD * 1.1, y2), xytext=(x1 + RAD * 1.1, y1),
                            arrowprops=dict(arrowstyle='-|>', color='#E67E22', lw=1.1, ls='--',
                                             connectionstyle=f'arc3,rad={rad}', alpha=0.8))

    # nodes (colored circles, value label beneath, tech label to the side)
    for n, (x, y) in pos.items():
        s, tech, r = n
        v = stage_output_value(s, tech, r, year)
        ax.add_patch(plt.Circle((x, y), RAD, color=REGION_COLOR[r], ec='white', lw=1.3, zorder=3))
        if v > min_flow:
            ax.text(x, y - RAD - 0.13, f'{v:.0f}', fontsize=7, color='#222', ha='center', va='top', zorder=4)
        if s == STAGE_ORDER[0]:
            ax.text(x - RAD - 0.12, y, tech, fontsize=7.5, ha='right', va='center', color='#555')
        elif s == STAGE_ORDER[-1]:
            ax.text(x + RAD + 0.12, y, tech, fontsize=7.5, ha='left', va='center', color='#555')
        else:
            ax.text(x, y + RAD + 0.10, tech, fontsize=6.5, ha='center', va='bottom', color='#555')

    for s in STAGE_ORDER:
        ax.text(stage_x[s], n_rows - 0.4, STAGE_LABEL[s], fontsize=12, fontweight='bold', ha='center')
    ax.set_title(f'Lithium Supply Chain Network (6 stages) — {year}', fontsize=15, fontweight='bold', pad=16)

    legend = [mpatches.Patch(color=REGION_COLOR['US'], label='US'),
              mpatches.Patch(color=REGION_COLOR['CHN'], label='CHN'),
              mpatches.Patch(color=REGION_COLOR['AUS'], label='AUS'),
              Line2D([0], [0], color='#555', lw=1.4, label='Conversion (stage -> stage)'),
              Line2D([0], [0], color='#E67E22', lw=1.4, ls='--', label='Trade')]
    ax.legend(handles=legend, loc='upper center', bbox_to_anchor=(0.5, -0.04), ncol=5, frameon=False, fontsize=9)

    ax.set_xlim(-1.5, stage_x[STAGE_ORDER[-1]] + 1.5)
    ax.set_ylim(-1, n_rows)
    ax.axis('off')
    plt.tight_layout()
    if save:
        plt.savefig(save, dpi=130, bbox_inches='tight')
    plt.show()


for yr in [Y[0], 2030, 2035]:
    draw_network(yr, save=f'network_{yr}.png')